# Appraisal QC — Full Training Pipeline

**Environment**: `apprisal` conda env · MacBook Air M1 8 GB (local run)

| Step | Model | Task |
|------|-------|------|
| A | LayoutLM-base | Extract field values from PDF text + layout |
| B | DistilBERT | Classify addendum commentary quality |
| C | MobileNetV3-small | Classify photo type + observed condition |
| D | Rule engine | Apply OPUS QC rules deterministically |

> **8 GB note** — models load and run sequentially. MPS (Metal) is used when available. Batch sizes and gradient accumulation are tuned to avoid OOM.

---
## 0 · Environment check & package installation

In [ ]:
import sys, subprocess, platform
print(f'Python : {sys.version}')
print(f'Machine: {platform.machine()}')

DEPS = [
    'torch torchvision torchaudio',
    'transformers>=4.40',
    'accelerate>=0.27',
    'datasets>=2.19',
    'seqeval',
    'ipykernel',
]

for dep in DEPS:
    top = dep.split()[0].split('>')[0].split('=')[0].replace('-', '_')
    try:
        __import__(top)
    except ImportError:
        print(f'Installing: {dep}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *dep.split(), '-q'])

print('All dependencies present.')

In [ ]:
import torch

if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    print('Device: MPS (Apple Silicon)')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f'Device: CUDA — {torch.cuda.get_device_name(0)}')
else:
    DEVICE = torch.device('cpu')
    print('Device: CPU')

print(f'PyTorch: {torch.__version__}')

---
## 1 · Imports & configuration

In [ ]:
import os, json, re, pickle, warnings
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import fitz  # PyMuPDF
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from transformers import (
    LayoutLMTokenizerFast, LayoutLMForTokenClassification,
    DistilBertTokenizerFast, DistilBertForSequenceClassification,
    TrainingArguments, Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
)
from datasets import Dataset as HFDataset

warnings.filterwarnings('ignore', category=UserWarning)
print('Imports OK')

In [ ]:
# ── project paths ──────────────────────────────────────────────────────────
# notebook lives at ocr-service/googlecolab/train_process.ipynb
# so .. resolves to ocr-service/
OCR_ROOT     = Path('..').resolve()
DATA_DIR     = OCR_ROOT / 'data'
MODEL_DIR    = OCR_ROOT / 'training' / 'models'
CKPT_DIR     = OCR_ROOT / 'training' / 'saved_checkpoints'
PIPELINE_PATH= MODEL_DIR / 'qc_pipeline.pkl'

for d in [
    DATA_DIR / 'pdfs',
    DATA_DIR / 'annotations',
    DATA_DIR / 'photos' / 'train',
    DATA_DIR / 'photos' / 'val',
    CKPT_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print('Directories ready:')
for d in [DATA_DIR, MODEL_DIR, CKPT_DIR]:
    print(f'  {d}')

# ── hyper-parameters (tuned for 8 GB M1) ──────────────────────────────────
CFG = {
    # LayoutLM field extractor
    'layoutlm_model'      : 'microsoft/layoutlm-base-uncased',
    'layoutlm_epochs'     : 5,
    'layoutlm_batch'      : 2,       # small batch — MPS memory
    'layoutlm_grad_accum' : 8,       # effective batch = 16
    'layoutlm_lr'         : 5e-5,
    'layoutlm_max_len'    : 512,
    # DistilBERT commentary classifier
    'distilbert_model'    : 'distilbert-base-uncased',
    'distilbert_epochs'   : 4,
    'distilbert_batch'    : 8,
    'distilbert_grad_accum': 4,      # effective batch = 32
    'distilbert_lr'       : 3e-5,
    'distilbert_max_len'  : 256,
    # MobileNetV3 photo classifier
    'photo_epochs'        : 10,
    'photo_batch'         : 16,
    'photo_lr'            : 1e-3,
    'photo_img_size'      : 224,
    # General
    'seed'                : 42,
    'val_split'           : 0.2,
}
print('Config loaded.')

---
## 2 · PDF processing utilities

In [ ]:
DPI = 150  # 150 dpi is sufficient for form text; saves memory vs 300


def pdf_to_images(pdf_path, dpi=DPI):
    """Render every page as a PIL Image (RGB)."""
    doc = fitz.open(str(pdf_path))
    out = []
    for page in doc:
        mat = fitz.Matrix(dpi / 72, dpi / 72)
        pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
        out.append(Image.frombytes('RGB', (pix.width, pix.height), pix.samples))
    doc.close()
    return out


def pdf_to_words_and_boxes(pdf_path, norm=1000):
    """Return (words, boxes) with bounding boxes normalised 0-1000 for LayoutLM."""
    doc = fitz.open(str(pdf_path))
    words, boxes = [], []
    for page in doc:
        pw, ph = page.rect.width, page.rect.height
        for blk in page.get_text('words'):
            x0, y0, x1, y1, word = blk[0], blk[1], blk[2], blk[3], blk[4]
            words.append(word)
            boxes.append([
                max(0, min(norm, int(x0 / pw * norm))),
                max(0, min(norm, int(y0 / ph * norm))),
                max(0, min(norm, int(x1 / pw * norm))),
                max(0, min(norm, int(y1 / ph * norm))),
            ])
    doc.close()
    return words, boxes


def pdf_to_full_text(pdf_path):
    doc = fitz.open(str(pdf_path))
    text = '\n'.join(page.get_text() for page in doc)
    doc.close()
    return text


# smoke test
sample_pdfs = list((DATA_DIR / 'pdfs').glob('*.pdf'))
if sample_pdfs:
    w, b = pdf_to_words_and_boxes(sample_pdfs[0])
    print(f'Sample PDF: {len(w)} words extracted')
else:
    print('No PDFs in data/pdfs/ yet — add appraisal PDFs before training.')

---
## 3 · Annotation loading

Annotation JSONs live in `data/annotations/<report_id>.json` using the schema from the design doc.
The loader flattens every field into a single DataFrame row.

In [ ]:
SECTION_KEYS = [
    'subject_section', 'contract_section', 'neighborhood_section',
    'site_section', 'improvements_section', 'sales_comparison_section',
    'reconciliation_section', 'cost_approach_section',
    'addendum_commentary_section', 'photo_section', 'sketch_section',
    'maps_section', 'signature_section', 'uspap_addendum_section',
    'market_conditions_addendum_1004mc',
]


def _flatten(sec_name, data, prefix=''):
    flat = {}
    for k, v in data.items():
        key = f'{prefix}__{k}' if prefix else f'{sec_name}__{k}'
        if isinstance(v, dict):
            flat.update(_flatten(sec_name, v, prefix=key))
        else:
            flat[key] = v
    return flat


def load_annotation(json_path):
    with open(json_path) as f:
        ann = json.load(f)
    row = {
        'report_id'    : ann.get('report_id'),
        'file_name'    : ann.get('file_name'),
        'form_type'    : ann.get('form_type'),
        'loan_type'    : ann.get('loan_type'),
        'effective_date': ann.get('effective_date'),
    }
    for sec in SECTION_KEYS:
        if sec in ann and isinstance(ann[sec], dict):
            row.update(_flatten(sec, ann[sec]))
    if 'overall_qc_summary' in ann:
        qc = ann['overall_qc_summary']
        row['qc_total'] = qc.get('total_rules_checked', 0)
        row['qc_pass']  = qc.get('pass', 0)
        row['qc_flags'] = qc.get('flag_for_review', 0)
    return row


def load_all_annotations(ann_dir):
    rows = []
    for jf in sorted(Path(ann_dir).glob('*.json')):
        try:
            rows.append(load_annotation(jf))
        except Exception as e:
            print(f'  SKIP {jf.name}: {e}')
    if not rows:
        print('No annotation JSONs found in data/annotations/')
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    print(f'Loaded {len(df)} annotations, {len(df.columns)} columns.')
    return df


df_ann = load_all_annotations(DATA_DIR / 'annotations')
if not df_ann.empty:
    display(df_ann.head(2))

In [ ]:
# ── build commentary training examples ────────────────────────────────────
COMM_LABEL_MAP = {'SPECIFIC': 0, 'GENERIC': 1, 'CANNED': 2, 'MISSING_REASONING': 3}

comm_rows = []
if not df_ann.empty:
    finding_cols = [c for c in df_ann.columns
                    if 'addendum' in c.lower() and c.endswith('__finding')]
    for _, row in df_ann.iterrows():
        for col in finding_cols:
            text = row.get(col)
            if not isinstance(text, str) or len(text) < 30:
                continue
            status_col = col.replace('__finding', '__status')
            status = row.get(status_col, 'PASS')
            label = (
                'GENERIC'           if status == 'FLAG' else
                'MISSING_REASONING' if status == 'FAIL' else
                'SPECIFIC'
            )
            comm_rows.append({'text': text, 'label': COMM_LABEL_MAP[label]})

df_comm = pd.DataFrame(comm_rows) if comm_rows else pd.DataFrame(columns=['text', 'label'])
print(f'Commentary samples: {len(df_comm)}')
if not df_comm.empty:
    print(df_comm['label'].value_counts())

---
## 4 · Dataset split (80 / 20)

In [ ]:
np.random.seed(CFG['seed'])


def split_df(df, val_frac=CFG['val_split'], stratify_col=None):
    if df.empty or len(df) < 4:
        print('Not enough data to split.')
        return df, df.iloc[:0].copy()
    strat = df[stratify_col] if stratify_col and stratify_col in df.columns else None
    tr, va = train_test_split(df, test_size=val_frac, random_state=CFG['seed'], stratify=strat)
    print(f'Train: {len(tr)}  Val: {len(va)}')
    return tr.reset_index(drop=True), va.reset_index(drop=True)


df_ann_tr, df_ann_va = split_df(df_ann) if not df_ann.empty else (pd.DataFrame(), pd.DataFrame())
df_comm_tr, df_comm_va = split_df(df_comm, stratify_col='label') if not df_comm.empty else (pd.DataFrame(), pd.DataFrame())

---
## 5 · Model A — Field extractor (LayoutLM-base, token classification)

LayoutLM takes words + bounding boxes from PyMuPDF and predicts BIO tags for 67 named fields.

> `batch_size=2` + `gradient_accumulation=8` → effective batch 16, stays within 8 GB.

In [ ]:
FIELD_NAMES = [
    'property_address', 'borrower_name', 'owner_of_public_record', 'county',
    'legal_description', 'assessors_parcel_number', 'tax_year', 'real_estate_taxes',
    'neighborhood_name', 'map_reference', 'census_tract', 'occupant_status',
    'special_assessments', 'pud_checkbox', 'hoa_dues', 'property_rights',
    'assignment_type', 'lender_name', 'lender_address', 'prior_listing',
    'contract_analyzed', 'sale_type', 'contract_price', 'contract_date',
    'seller_is_owner_of_record', 'financial_assistance_checkbox', 'financial_assistance_amount',
    'location_checkbox', 'built_up_checkbox', 'growth_checkbox',
    'property_values_trend', 'demand_supply', 'marketing_time',
    'price_range_low', 'price_range_high', 'price_predominant',
    'land_use_one_unit_percent', 'dimensions', 'site_area', 'site_shape', 'view',
    'zoning_classification', 'zoning_compliance', 'highest_and_best_use',
    'fema_flood_zone', 'fema_map_number', 'fema_map_date',
    'stories', 'existing_proposed', 'year_built', 'effective_age',
    'foundation', 'condition_rating',
    'above_grade_total_rooms', 'above_grade_bedrooms', 'above_grade_baths',
    'gross_living_area',
    'sales_comparison_value', 'cost_approach_value', 'final_value',
    'site_value', 'cost_new', 'depreciation', 'indicated_value_cost',
    'appraiser_name', 'date_of_signature', 'state_certification_number', 'expiration_date',
]

BIO_LABELS = ['O'] + [f'B-{f}' for f in FIELD_NAMES] + [f'I-{f}' for f in FIELD_NAMES]
LABEL2ID   = {lbl: i for i, lbl in enumerate(BIO_LABELS)}
ID2LABEL   = {i: lbl for lbl, i in LABEL2ID.items()}
print(f'BIO labels: {len(BIO_LABELS)}  (O + {len(FIELD_NAMES)} B + {len(FIELD_NAMES)} I)')

In [ ]:
layoutlm_tokenizer = LayoutLMTokenizerFast.from_pretrained(CFG['layoutlm_model'])


def annotation_to_spans(ann_row):
    """Return {field_name: value_string} for fields present in the annotation row."""
    spans = {}
    for key, val in ann_row.items():
        if key.endswith('__extracted_value') and isinstance(val, str) and val.strip():
            parts = key.split('__')
            # key format: section__field__extracted_value  (parts[-2] = field)
            field = parts[-2]
            if field in FIELD_NAMES:
                spans[field] = val.strip()
    return spans


def make_layoutlm_example(pdf_path, annotation_row, max_len=CFG['layoutlm_max_len']):
    words, boxes = pdf_to_words_and_boxes(pdf_path)
    spans = annotation_to_spans(annotation_row)

    # BIO label each word (greedy, case-insensitive)
    word_labels = ['O'] * len(words)
    for field, value in spans.items():
        val_words = value.split()
        vlen = len(val_words)
        for i in range(len(words) - vlen + 1):
            if [w.lower() for w in words[i:i+vlen]] == [v.lower() for v in val_words]:
                word_labels[i] = f'B-{field}'
                for j in range(1, vlen):
                    word_labels[i+j] = f'I-{field}'
                break

    enc = layoutlm_tokenizer(
        words, boxes=boxes,
        is_split_into_words=True,
        max_length=max_len,
        truncation=True,
        padding='max_length',
    )
    word_ids  = enc.word_ids()
    label_ids, prev_wid = [], None
    for wid in word_ids:
        if wid is None:
            label_ids.append(-100)
        elif wid != prev_wid:
            label_ids.append(LABEL2ID.get(word_labels[wid], 0))
        else:
            lbl = word_labels[wid]
            if lbl.startswith('B-'):
                lbl = 'I-' + lbl[2:]
            label_ids.append(LABEL2ID.get(lbl, -100))
        prev_wid = wid

    return {
        'input_ids'      : enc['input_ids'],
        'attention_mask' : enc['attention_mask'],
        'token_type_ids' : enc.get('token_type_ids', [0]*max_len),
        'bbox'           : enc['bbox'],
        'labels'         : label_ids,
    }


def build_layoutlm_dataset(ann_df, pdf_dir):
    if ann_df.empty:
        return None
    records = []
    for _, row in tqdm(ann_df.iterrows(), total=len(ann_df), desc='Encoding PDFs'):
        pdf_path = Path(pdf_dir) / str(row.get('file_name', ''))
        if not pdf_path.exists():
            continue
        records.append(make_layoutlm_example(pdf_path, row.to_dict()))
    if not records:
        print('No matching PDFs found — add PDFs to data/pdfs/')
        return None
    print(f'Built {len(records)} LayoutLM examples.')
    return HFDataset.from_list(records)


train_lm_ds = build_layoutlm_dataset(df_ann_tr, DATA_DIR / 'pdfs')
val_lm_ds   = build_layoutlm_dataset(df_ann_va, DATA_DIR / 'pdfs')

In [ ]:
from seqeval.metrics import f1_score as seq_f1


def compute_ner_metrics(p):
    preds_flat, labels_flat = p
    pred_ids = np.argmax(preds_flat, axis=-1)
    true_seqs, pred_seqs = [], []
    for pred_row, label_row in zip(pred_ids, labels_flat):
        ts, ps = [], []
        for pid, lid in zip(pred_row, label_row):
            if lid == -100:
                continue
            ts.append(ID2LABEL[lid])
            ps.append(ID2LABEL[pid])
        true_seqs.append(ts)
        pred_seqs.append(ps)
    return {'f1': seq_f1(true_seqs, pred_seqs)}


def train_field_extractor(train_ds, val_ds):
    if train_ds is None or val_ds is None:
        print('No dataset — skipping LayoutLM training.')
        return None

    model = LayoutLMForTokenClassification.from_pretrained(
        CFG['layoutlm_model'],
        num_labels=len(BIO_LABELS),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    ).to(DEVICE)

    ckpt = str(CKPT_DIR / 'layoutlm_field_extractor')
    args = TrainingArguments(
        output_dir                  = ckpt,
        num_train_epochs            = CFG['layoutlm_epochs'],
        per_device_train_batch_size = CFG['layoutlm_batch'],
        per_device_eval_batch_size  = CFG['layoutlm_batch'],
        gradient_accumulation_steps = CFG['layoutlm_grad_accum'],
        learning_rate               = CFG['layoutlm_lr'],
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'f1',
        logging_steps               = 10,
        seed                        = CFG['seed'],
        use_mps_device              = (DEVICE.type == 'mps'),
        fp16                        = False,
        bf16                        = False,
        dataloader_num_workers      = 0,   # avoid fork issues on macOS
        report_to                   = 'none',
    )

    collator = DataCollatorForTokenClassification(layoutlm_tokenizer)
    trainer  = Trainer(
        model           = model,
        args            = args,
        train_dataset   = train_ds,
        eval_dataset    = val_ds,
        compute_metrics = compute_ner_metrics,
        data_collator   = collator,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
    )
    trainer.train()
    trainer.save_model(ckpt + '/best')
    print(f'LayoutLM saved: {ckpt}/best')
    return model


layoutlm_model = train_field_extractor(train_lm_ds, val_lm_ds)

In [ ]:
def extract_fields(pdf_path, model=None, tokenizer=None):
    """Run field extractor on one PDF. Returns {field_name: value_string}."""
    if model is None:
        return {}
    tok = tokenizer or layoutlm_tokenizer
    model.eval()
    words, boxes = pdf_to_words_and_boxes(pdf_path)
    enc = tok(
        words, boxes=boxes,
        is_split_into_words=True,
        max_length=CFG['layoutlm_max_len'],
        truncation=True,
        padding='max_length',
        return_tensors='pt',
    )
    enc_dev = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc_dev).logits
    pred_ids = logits.argmax(-1)[0].tolist()
    word_ids = enc.word_ids()

    extracted, current_field, current_tokens, prev_wid = {}, None, [], None
    for pid, wid in zip(pred_ids, word_ids):
        if wid is None:
            continue
        label = ID2LABEL.get(pid, 'O')
        word  = words[wid] if wid < len(words) else ''
        if label.startswith('B-'):
            if current_field:
                extracted[current_field] = ' '.join(current_tokens)
            current_field  = label[2:]
            current_tokens = [word] if wid != prev_wid else []
        elif label.startswith('I-') and current_field and wid != prev_wid:
            current_tokens.append(word)
        elif label == 'O':
            if current_field:
                extracted[current_field] = ' '.join(current_tokens)
            current_field, current_tokens = None, []
        prev_wid = wid
    if current_field:
        extracted[current_field] = ' '.join(current_tokens)
    return extracted


print('extract_fields() ready.')

---
## 6 · Model B — Commentary classifier (DistilBERT)

Classifies each addendum paragraph as `SPECIFIC` / `GENERIC` / `CANNED` / `MISSING_REASONING`.

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

COMM_LABELS = ['SPECIFIC', 'GENERIC', 'CANNED', 'MISSING_REASONING']
id2comm     = {i: l for i, l in enumerate(COMM_LABELS)}
comm2id     = {l: i for i, l in id2comm.items()}
comm_tokenizer = DistilBertTokenizerFast.from_pretrained(CFG['distilbert_model'])


class CommentaryDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts  = df['text'].tolist()
        self.labels = df['label'].tolist()
        self.tok    = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids'     : enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels'        : torch.tensor(self.labels[idx], dtype=torch.long),
        }


def train_commentary_classifier(df_tr, df_va):
    if df_tr.empty:
        print('No commentary data — skipping.')
        return None

    tr_dl = DataLoader(CommentaryDataset(df_tr, comm_tokenizer, CFG['distilbert_max_len']),
                       batch_size=CFG['distilbert_batch'], shuffle=True, num_workers=0)
    va_dl = DataLoader(CommentaryDataset(df_va, comm_tokenizer, CFG['distilbert_max_len']),
                       batch_size=CFG['distilbert_batch'], shuffle=False, num_workers=0)

    model = DistilBertForSequenceClassification.from_pretrained(
        CFG['distilbert_model'], num_labels=len(COMM_LABELS),
        id2label=id2comm, label2id=comm2id,
    ).to(DEVICE)

    optimizer = AdamW(model.parameters(), lr=CFG['distilbert_lr'])
    scheduler = CosineAnnealingLR(optimizer, T_max=CFG['distilbert_epochs'])
    accum     = CFG['distilbert_grad_accum']
    best_acc, best_state = 0.0, None

    for epoch in range(CFG['distilbert_epochs']):
        model.train()
        optimizer.zero_grad()
        total_loss = 0
        for step, batch in enumerate(tqdm(tr_dl, desc=f'Comm epoch {epoch+1}')):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            loss  = model(**batch).loss / accum
            loss.backward()
            total_loss += loss.item() * accum
            if (step + 1) % accum == 0:
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()
        scheduler.step()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch in va_dl:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                preds = model(**batch).logits.argmax(-1)
                correct += (preds == batch['labels']).sum().item()
                total   += len(batch['labels'])
        acc = correct / total if total else 0
        print(f'  Epoch {epoch+1}: loss={total_loss/len(tr_dl):.3f}  val_acc={acc:.3f}')
        if acc > best_acc:
            best_acc  = acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    ckpt = str(CKPT_DIR / 'distilbert_commentary')
    model.save_pretrained(ckpt)
    comm_tokenizer.save_pretrained(ckpt)
    print(f'Commentary model saved: {ckpt}  (best val_acc={best_acc:.3f})')
    return model


if DEVICE.type == 'mps':
    torch.mps.empty_cache()
comm_model = train_commentary_classifier(df_comm_tr, df_comm_va)

In [ ]:
def classify_commentary(text, model=None, tokenizer=None):
    """Return {label, confidence} for one addendum paragraph."""
    if model is None or not text.strip():
        return {'label': 'UNKNOWN', 'confidence': 0.0}
    tok = tokenizer or comm_tokenizer
    model.eval()
    enc = tok(text, max_length=CFG['distilbert_max_len'],
               truncation=True, padding='max_length', return_tensors='pt')
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
    probs  = torch.softmax(logits, -1)[0].tolist()
    top_id = int(logits.argmax().item())
    return {'label': id2comm[top_id], 'confidence': round(probs[top_id], 4)}


print('classify_commentary() ready.')

---
## 7 · Model C — Photo classifier (MobileNetV3-small)

Labels: `front_exterior_C1` · `front_exterior_C2` · `front_exterior_C3` · `rear_exterior` · `street_scene` · `interior_kitchen` · `interior_bathroom` · `interior_bedroom` · `interior_living` · `comparable_photo` · `aerial_map` · `sketch`

Put training images in `data/photos/train/<label>/` and validation images in `data/photos/val/<label>/`.

In [ ]:
from torchvision.datasets import ImageFolder

PHOTO_CLASSES = [
    'front_exterior_C1', 'front_exterior_C2', 'front_exterior_C3',
    'rear_exterior', 'street_scene',
    'interior_kitchen', 'interior_bathroom', 'interior_bedroom', 'interior_living',
    'comparable_photo', 'aerial_map', 'sketch',
]
for cls in PHOTO_CLASSES:
    (DATA_DIR / 'photos' / 'train' / cls).mkdir(parents=True, exist_ok=True)
    (DATA_DIR / 'photos' / 'val'   / cls).mkdir(parents=True, exist_ok=True)

IMG_MEAN = [0.485, 0.456, 0.406]
IMG_STD  = [0.229, 0.224, 0.225]

train_tfm = transforms.Compose([
    transforms.Resize((CFG['photo_img_size'], CFG['photo_img_size'])),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMG_MEAN, IMG_STD),
])
val_tfm = transforms.Compose([
    transforms.Resize((CFG['photo_img_size'], CFG['photo_img_size'])),
    transforms.ToTensor(),
    transforms.Normalize(IMG_MEAN, IMG_STD),
])

has_train_photos = any(
    list((DATA_DIR / 'photos' / 'train' / cls).glob('*.*'))
    for cls in PHOTO_CLASSES
)

if has_train_photos:
    train_photo_ds = ImageFolder(str(DATA_DIR / 'photos' / 'train'), transform=train_tfm)
    val_photo_ds   = ImageFolder(str(DATA_DIR / 'photos' / 'val'),   transform=val_tfm)
    train_photo_dl = DataLoader(train_photo_ds, batch_size=CFG['photo_batch'], shuffle=True,  num_workers=0)
    val_photo_dl   = DataLoader(val_photo_ds,   batch_size=CFG['photo_batch'], shuffle=False, num_workers=0)
    print(f'Photo train: {len(train_photo_ds)} images, {len(train_photo_ds.classes)} classes')
else:
    train_photo_dl = val_photo_dl = None
    print('No photos yet — add images to data/photos/train/<class>/ first.')

In [ ]:
def train_photo_classifier(train_dl, val_dl):
    if train_dl is None:
        print('No photo data — skipping.')
        return None

    model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
    in_f  = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_f, len(PHOTO_CLASSES))
    model = model.to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG['photo_lr'])
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.5)
    best_acc, best_state = 0.0, None

    for epoch in range(CFG['photo_epochs']):
        model.train()
        for imgs, labels in tqdm(train_dl, desc=f'Photo epoch {epoch+1}'):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(imgs), labels).backward()
            optimizer.step()
        scheduler.step()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_dl:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                correct += (model(imgs).argmax(1) == labels).sum().item()
                total   += len(labels)
        acc = correct / total if total else 0
        print(f'  Epoch {epoch+1}: val_acc={acc:.3f}')
        if acc > best_acc:
            best_acc   = acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    ckpt = CKPT_DIR / 'mobilenet_photo.pt'
    torch.save({'state_dict': model.state_dict(), 'classes': PHOTO_CLASSES}, str(ckpt))
    print(f'Photo model saved: {ckpt}  (best val_acc={best_acc:.3f})')
    return model


if DEVICE.type == 'mps':
    torch.mps.empty_cache()
photo_model = train_photo_classifier(train_photo_dl, val_photo_dl)

In [ ]:
def classify_photo(img, model=None):
    """Return {label, confidence} for one PIL Image."""
    if model is None:
        return {'label': 'UNKNOWN', 'confidence': 0.0}
    model.eval()
    tensor = val_tfm(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(tensor)
    probs  = torch.softmax(logits, -1)[0].tolist()
    top_id = int(logits.argmax().item())
    return {'label': PHOTO_CLASSES[top_id], 'confidence': round(probs[top_id], 4)}


print('classify_photo() ready.')

---
## 8 · Rule engine (deterministic OPUS QC)

In [ ]:
import sys as _sys
import re
_ocr_root = str(OCR_ROOT)
if _ocr_root not in _sys.path:
    _sys.path.insert(0, _ocr_root)

PASS_S, FLAG_S, FAIL_S, VERIFY_S = 'PASS', 'FLAG', 'FAIL', 'NEEDS_VERIFICATION'


def _f(rule, status, text=None):
    return {'rule': rule, 'status': status, 'finding': text}


def _num(val):
    try:
        return float(str(val).replace(',', '').replace('$', ''))
    except (ValueError, TypeError):
        return 0.0


# ── section checks ─────────────────────────────────────────────────────────

def _subject(f):
    out = []
    if not f.get('property_address'):
        out.append(_f('S-1', FAIL_S, 'Property address not extracted.'))
    else:
        out.append(_f('S-1', VERIFY_S, 'Verify address against engagement letter.'))

    ct = f.get('census_tract', '')
    if ct and not re.match(r'^\d{4}\.\d{2}$', str(ct)):
        out.append(_f('S-6', FLAG_S, f"Census tract '{ct}' should be XXXX.XX format."))
    elif ct:
        out.append(_f('S-6', PASS_S))

    appr  = _num(f.get('final_value', 0))
    contr = _num(f.get('contract_price', 0))
    if appr and contr:
        pct = abs(appr - contr) / contr
        if pct >= 0.05:
            out.append(_f('S-12', FLAG_S,
                f"Value ${appr:,.0f} is {pct*100:.1f}% "
                f"{'above' if appr > contr else 'below'} contract ${contr:,.0f}. "
                "Comment required."))
    return out


def _contract(f):
    out = []
    if not f.get('contract_price'):
        out.append(_f('C-2', VERIFY_S, 'Contract price not found.'))
    else:
        out.append(_f('C-2', VERIFY_S, 'Verify contract price against purchase agreement.'))
    fa_cb  = str(f.get('financial_assistance_checkbox', '')).lower()
    fa_amt = _num(f.get('financial_assistance_amount', 0))
    if fa_cb in ('yes', 'true', '1') and fa_amt == 0:
        out.append(_f('C-4', FLAG_S, 'Financial assistance Yes but amount is $0.'))
    return out


def _neighborhood(f):
    out = []
    appr = _num(f.get('final_value', 0))
    low  = _num(f.get('price_range_low', 0)) * 1000
    high = _num(f.get('price_range_high', 0)) * 1000
    if appr and low and high and not (low <= appr <= high):
        out.append(_f('N-3', FLAG_S, f'Value ${appr:,.0f} outside range ${low:,.0f}–${high:,.0f}.'))
    elif appr and low and high:
        out.append(_f('N-3', PASS_S))
    one_unit = _num(f.get('land_use_one_unit_percent', 0))
    if one_unit and one_unit < 100:
        out.append(_f('N-4', FLAG_S, f'One-unit {one_unit}% — verify all categories total 100%.'))
    return out


def _site(f):
    out = []
    if 'irregular' in str(f.get('site_shape', '')).lower():
        out.append(_f('ST-3', FLAG_S, 'Irregular shape — plat map with subject marked required.'))
    fema = f.get('fema_map_number', '')
    out.append(_f('ST-8', PASS_S if fema else FLAG_S,
                  None if fema else 'FEMA map number not found.'))
    return out


def _improvements(f):
    out = []
    cond = str(f.get('condition_rating', '')).upper()
    yr   = _num(f.get('year_built', 0))
    if cond == 'C1' and yr and yr < 2024:
        out.append(_f('I-9', FLAG_S, f'C1 condition for year built {yr:.0f} — verify new/never-occupied.'))
    elif cond:
        out.append(_f('I-9', PASS_S))
    gla = f.get('gross_living_area', '')
    out.append(_f('I-7', PASS_S if gla else FLAG_S,
                  None if gla else 'GLA not extracted.'))
    return out


def _sales_comparison(f):
    out = []
    appr = _num(f.get('final_value', 0))
    if appr and appr < 1_000_000:
        out.append(_f('SCA-2', FLAG_S,
            'Verify minimum 3 closed sales + 2 active listings provided (value < $1M).'))
    return out


def _reconciliation(f):
    out = []
    appr  = _num(f.get('final_value', 0))
    contr = _num(f.get('contract_price', 0))
    if appr and contr:
        pct = (appr - contr) / contr
        if abs(pct) >= 0.05:
            out.append(_f('R-2', FLAG_S,
                f'Value is {pct*100:+.1f}% vs contract — reconciliation must explain variance.'))
        else:
            out.append(_f('R-2', PASS_S))
    return out


def _addendum(comm_results):
    out = []
    labels = {r.get('label') for r in comm_results}
    if 'CANNED' in labels:
        out.append(_f('ADD-1', FLAG_S, 'Canned/boilerplate commentary detected.'))
    if 'GENERIC' in labels:
        out.append(_f('ADD-2', FLAG_S, 'Generic commentary detected — provide specific analysis.'))
    if 'MISSING_REASONING' in labels:
        out.append(_f('ADD-1', FLAG_S, 'Commentary lacks WHY — reasoning must be stated.'))
    if not out:
        out.append(_f('ADD-1', PASS_S))
    return out


def _photos(photo_results):
    out = []
    found = {r.get('label', '') for r in photo_results}
    if not any(l.startswith('front_exterior') for l in found):
        out.append(_f('PH-1', FLAG_S, 'Front exterior photo not detected.'))
    else:
        out.append(_f('PH-1', PASS_S))
    for lbl, rule in [('rear_exterior', 'PH-1'), ('street_scene', 'PH-1')]:
        out.append(_f(rule, PASS_S if lbl in found else FLAG_S,
                      None if lbl in found else f'{lbl.replace("_", " ").title()} photo not detected.'))
    return out


def _signature(f):
    out = []
    from datetime import datetime
    exp = f.get('expiration_date', '')
    eff = f.get('effective_date', '')
    if exp and eff:
        try:
            if datetime.strptime(exp, '%m/%d/%Y') < datetime.strptime(eff, '%m/%d/%Y'):
                out.append(_f('SIG-2', FAIL_S, f'License expired {exp} before effective date {eff}.'))
            else:
                out.append(_f('SIG-2', PASS_S))
        except ValueError:
            out.append(_f('SIG-2', VERIFY_S, 'Could not parse dates for expiry check.'))
    return out


def _summary(findings):
    all_f = [f for sec in findings.values() if isinstance(sec, list) for f in sec]
    return {
        'total' : len(all_f),
        'pass'  : sum(1 for f in all_f if f['status'] == PASS_S),
        'flag'  : sum(1 for f in all_f if f['status'] == FLAG_S),
        'fail'  : sum(1 for f in all_f if f['status'] == FAIL_S),
        'verify': sum(1 for f in all_f if f['status'] == VERIFY_S),
    }


def rule_engine(fields, commentary_results, photo_results,
                loan_type='Conventional', form_type='1004'):
    findings = {
        'subject'         : _subject(fields),
        'contract'        : _contract(fields),
        'neighborhood'    : _neighborhood(fields),
        'site'            : _site(fields),
        'improvements'    : _improvements(fields),
        'sales_comparison': _sales_comparison(fields),
        'reconciliation'  : _reconciliation(fields),
        'addendum'        : _addendum(commentary_results),
        'photos'          : _photos(photo_results),
        'signature'       : _signature(fields),
    }
    findings['summary'] = _summary(findings)
    return findings


print('Rule engine ready.')

---
## 9 · AppraisalQCPipeline class

In [ ]:
class AppraisalQCPipeline:
    """
    Single entry-point: pipeline.run(pdf_path) -> QC findings JSON.
    Wraps LayoutLM + DistilBERT + MobileNetV3 + rule engine.
    """

    def __init__(self, layoutlm_model=None, layoutlm_tokenizer=None,
                 comm_model=None, comm_tokenizer=None,
                 photo_model=None, photo_classes=None,
                 device=None, cfg=None):
        self.lm_model      = layoutlm_model
        self.lm_tok        = layoutlm_tokenizer
        self.comm_model    = comm_model
        self.comm_tok      = comm_tokenizer
        self.photo_model   = photo_model
        self.photo_classes = photo_classes or PHOTO_CLASSES
        self.device        = device or DEVICE
        self.cfg           = cfg or CFG

    # ── public API ──────────────────────────────────────────────────────────
    def run(self, pdf_path, loan_type='Conventional', form_type='1004'):
        pdf_path = Path(pdf_path)
        assert pdf_path.exists(), f'PDF not found: {pdf_path}'

        # 1. field extraction
        fields = extract_fields(pdf_path, self.lm_model, self.lm_tok)
        fields['effective_date'] = fields.get('effective_date', '')

        # 2. commentary classification
        full_text = pdf_to_full_text(pdf_path)
        paragraphs = [p.strip() for p in full_text.split('\n\n') if len(p.strip()) > 50][:20]
        comm_results = [classify_commentary(p, self.comm_model, self.comm_tok)
                        for p in paragraphs]

        # 3. photo classification (only pages that look like photos)
        pages = pdf_to_images(pdf_path)
        photo_results = []
        for page_img in pages:
            arr = np.array(page_img.convert('L'))
            if arr.std() > 30:   # high variance = likely photo/image page
                photo_results.append(classify_photo(page_img, self.photo_model))

        # 4. rule engine
        qc = rule_engine(fields, comm_results, photo_results,
                         loan_type=loan_type, form_type=form_type)

        return {
            'pdf'               : pdf_path.name,
            'loan_type'         : loan_type,
            'form_type'         : form_type,
            'extracted_fields'  : fields,
            'commentary_results': comm_results,
            'photo_results'     : photo_results,
            'qc_findings'       : qc,
        }

    # ── save ────────────────────────────────────────────────────────────────
    def save(self, path):
        path = Path(path)
        bdir = path.with_suffix('.bundle')
        bdir.mkdir(parents=True, exist_ok=True)

        if self.lm_model:
            self.lm_model.save_pretrained(str(bdir / 'layoutlm'))
            self.lm_tok.save_pretrained(str(bdir / 'layoutlm'))
        if self.comm_model:
            self.comm_model.save_pretrained(str(bdir / 'distilbert'))
            self.comm_tok.save_pretrained(str(bdir / 'distilbert'))
        if self.photo_model:
            torch.save({'state_dict': self.photo_model.state_dict(),
                        'classes'   : self.photo_classes}, str(bdir / 'photo.pt'))
        with open(bdir / 'config.json', 'w') as fh:
            json.dump(self.cfg, fh, indent=2)

        with open(path, 'wb') as fh:
            pickle.dump({'bundle_dir': str(bdir), 'cfg': self.cfg,
                         'photo_classes': self.photo_classes}, fh)
        print(f'Pipeline saved → {path}')

    # ── load ────────────────────────────────────────────────────────────────
    @classmethod
    def load(cls, path, device=None):
        dev = device or DEVICE
        with open(path, 'rb') as fh:
            shell = pickle.load(fh)
        bdir       = Path(shell['bundle_dir'])
        cfg        = shell.get('cfg', CFG)
        ph_classes = shell.get('photo_classes', PHOTO_CLASSES)

        lm_model = lm_tok = None
        if (bdir / 'layoutlm').exists():
            lm_tok   = LayoutLMTokenizerFast.from_pretrained(str(bdir / 'layoutlm'))
            lm_model = LayoutLMForTokenClassification.from_pretrained(
                str(bdir / 'layoutlm')).to(dev)

        c_model = c_tok = None
        if (bdir / 'distilbert').exists():
            c_tok   = DistilBertTokenizerFast.from_pretrained(str(bdir / 'distilbert'))
            c_model = DistilBertForSequenceClassification.from_pretrained(
                str(bdir / 'distilbert')).to(dev)

        p_model = None
        if (bdir / 'photo.pt').exists():
            ckpt  = torch.load(str(bdir / 'photo.pt'), map_location=dev)
            p_model = models.mobilenet_v3_small(weights=None)
            p_model.classifier[-1] = nn.Linear(
                p_model.classifier[-1].in_features, len(ph_classes))
            p_model.load_state_dict(ckpt['state_dict'])
            p_model = p_model.to(dev)

        return cls(lm_model, lm_tok, c_model, c_tok, p_model, ph_classes, dev, cfg)


print('AppraisalQCPipeline class ready.')

---
## 10 · Assemble, save & round-trip test

In [ ]:
pipeline = AppraisalQCPipeline(
    layoutlm_model     = layoutlm_model,
    layoutlm_tokenizer = layoutlm_tokenizer,
    comm_model         = comm_model,
    comm_tokenizer     = comm_tokenizer,
    photo_model        = photo_model,
    photo_classes      = PHOTO_CLASSES,
    device             = DEVICE,
    cfg                = CFG,
)

pipeline.save(PIPELINE_PATH)

# round-trip
pipeline_loaded = AppraisalQCPipeline.load(PIPELINE_PATH)
print('Round-trip load OK.')

---
## 11 · Evaluation — test on unseen appraisals

Compare pipeline output against human annotation JSONs.  
The gap table shows which fields to focus on in the next training round.

In [ ]:
def evaluate_against_annotation(pdf_path, ann_json_path, pipeline):
    with open(ann_json_path) as fh:
        human_ann = json.load(fh)
    result = pipeline.run(pdf_path)
    rows = []
    for sec_key, sec_data in human_ann.items():
        if not isinstance(sec_data, dict):
            continue
        for field_key, field_data in sec_data.items():
            if not isinstance(field_data, dict):
                continue
            human_val = field_data.get('extracted_value', '')
            ml_val    = result['extracted_fields'].get(field_key, '')
            rows.append({
                'section'      : sec_key,
                'field'        : field_key,
                'human_value'  : human_val,
                'ml_value'     : ml_val,
                'value_match'  : str(human_val).lower().strip() == str(ml_val).lower().strip(),
                'human_status' : field_data.get('status', ''),
                'human_finding': field_data.get('finding', ''),
            })
    return pd.DataFrame(rows)


# use last 3 PDFs as unseen test set
test_pdfs = sorted((DATA_DIR / 'pdfs').glob('*.pdf'))[-3:]
gap_dfs   = []

for pdf_path in test_pdfs:
    ann_path = DATA_DIR / 'annotations' / (pdf_path.stem + '.json')
    if ann_path.exists():
        print(f'\nEvaluating: {pdf_path.name}')
        gap_df  = evaluate_against_annotation(pdf_path, ann_path, pipeline_loaded)
        gap_dfs.append(gap_df)
        matched = gap_df['value_match'].sum()
        total   = len(gap_df)
        print(f'  Field match: {matched}/{total} = {matched/total*100:.1f}%')
    else:
        print(f'  No annotation for {pdf_path.name}')

if gap_dfs:
    all_gaps = pd.concat(gap_dfs, ignore_index=True)
    missed   = all_gaps[~all_gaps['value_match']]
    print(f'\nTotal mismatches: {len(missed)}')
    display(missed[['section', 'field', 'human_value', 'ml_value']].head(30))
else:
    print('Add PDF + annotation JSON pairs to data/ to run evaluation.')

In [ ]:
# ── gap report → drives next training round ────────────────────────────────
if gap_dfs:
    miss_freq = (
        missed
        .groupby(['section', 'field'])
        .size()
        .reset_index(name='miss_count')
        .sort_values('miss_count', ascending=False)
    )
    print('Top fields to fix in next training round:')
    display(miss_freq.head(20))

    gap_path = MODEL_DIR / 'gap_report.csv'
    miss_freq.to_csv(gap_path, index=False)
    print(f'Gap report saved: {gap_path}')

In [ ]:
# ── quick demo on any available PDF ───────────────────────────────────────
demo_pdfs = sorted((DATA_DIR / 'pdfs').glob('*.pdf'))
if demo_pdfs:
    demo = pipeline_loaded.run(demo_pdfs[0])
    print(json.dumps(demo['qc_findings']['summary'], indent=2))
    print('\nExtracted fields (first 10):')
    for k, v in list(demo['extracted_fields'].items())[:10]:
        print(f'  {k:40s}: {v}')
else:
    print('Place at least one PDF in data/pdfs/ for a live demo.')

---
## Appendix — directory layout & minimum data requirements

```
ocr-service/
  data/
    pdfs/                     ← appraisal PDFs  (<report_id>.pdf)
    annotations/              ← QC annotation JSONs  (<report_id>.json)
    photos/
      train/<class>/          ← labelled photos for Model C training
      val/<class>/
  training/
    models/                   ← final model files + gap_report.csv
    saved_checkpoints/        ← per-epoch HuggingFace + torch checkpoints
  googlecolab/
    train_process.ipynb       ← this notebook
```

| Model | Minimum samples |
|-------|-----------------|
| LayoutLM field extractor | 30+ annotation JSONs + matching PDFs |
| DistilBERT commentary    | 200+ labelled paragraphs |
| MobileNetV3 photos       | 100+ images per class |

**Iteration loop**: run Section 11 after each training → `gap_report.csv` shows which fields missed most → add targeted annotation examples → retrain.